SOC612: Data Analytics for the Social Sciences - Solutions

Date: September 16, 2026

Author: R. Duerr

Task 1: Data Preparation

**1**.

In [ ]:
import pandas as pd

wvs = pd.read_csv("wvs6_7.csv")
wbdata = pd.read_excel("wbdata.xlsx")

**2.**

In [ ]:
wvs = (
    wvs.merge(
        wbdata[["Code", "Income group"]],
        how="left",
        left_on="B_COUNTRY_ALPHA",
        right_on="Code",
    )
    .drop(columns="Code")
)

wvslm = wvs.loc[wvs["Income group"] == "Lower middle income"].copy()
wvslm["B_COUNTRY_ALPHA"] = wvslm["B_COUNTRY_ALPHA"].astype("category")

**3.**


In [ ]:
wvslm = wvslm[[
    "B_COUNTRY_ALPHA", "Q260", "Q262", "Q270", "Q273", "Q274",
    "Q275", "Q288", "Q58", "Q59", "Q61", "Q62", "Q63",
]].rename(columns={
    "B_COUNTRY_ALPHA": "country", "Q260": "sex", "Q262": "age", "Q270": "ppl_hh",
    "Q273": "marital_stat", "Q274": "nmbr_chldrn", "Q275": "edu", "Q288": "inc",
    "Q58": "trust_family", "Q59": "trust_neighbors", "Q61": "trust_first",
    "Q62": "trust_religion", "Q63": "trust_nationality",
})

**4.**



In [ ]:
trust_cols = [
    "trust_family", "trust_neighbors", "trust_first","trust_religion",
    "trust_nationality",
]
wvslm[trust_cols] = wvslm[trust_cols].where(wvslm[trust_cols].between(1, 4))
wvslm[trust_cols] = wvslm[trust_cols] - 1
wvslm[trust_cols] = wvslm[trust_cols] * -1 + 3

**5.**

In [ ]:
wvslm["marital_stat"] = wvslm["marital_stat"].map({
    1: "married", 2: "married",
    3: "single", 4: "single", 5: "single", 6: "single",
})
wvslm["sex"] = wvslm["sex"].map({
    1: "male", 2: "female"})
wvslm["edu"] = wvslm["edu"].map({
    0: "low", 1: "low", 2: "low",
    3: "middle", 4: "middle",
    5: "high", 6: "high", 7: "high", 8: "high",
})

for col in ["marital_stat", "sex", "edu"]:
    wvslm[col] = wvslm[col].astype("category")

**6.**

In [ ]:
wvslm["age"] = wvslm["age"].mask(wvslm["age"] < 18)
wvslm["ppl_hh"] = wvslm["ppl_hh"].mask(wvslm["ppl_hh"] < 1)
wvslm["nmbr_chldrn"] = wvslm["nmbr_chldrn"].mask(wvslm["nmbr_chldrn"] < 0)
wvslm["inc"] = wvslm["inc"].mask(wvslm["inc"] < 1)

**7.**

In [ ]:
wvslm["trust_index"] = wvslm[trust_cols].sum(axis=1)

**8.**

In [ ]:
def normalize(x):
    return (x - x.min()) / (x.max() - x.min())

wvslm["trust_index_norm"] = normalize(wvslm["trust_index"])

# Bonus
country_means = wvslm.groupby("country", observed=True)[trust_cols].mean()
wvslm["trust_mean_sum"] = wvslm["country"].map(country_means.sum(axis=1))
wvslm["trust_index_diff"] = wvslm["trust_index"] - wvslm["trust_mean_sum"]

**9.**

In [ ]:
edu_num = wvslm["edu"].map({"low": 1, "middle": 2, "high": 3})

wvslm["index_ses"] = edu_num * wvslm["inc"]

# possible alternative: slightly diminishing return of higher income
wvslm["index_ses2"] = edu_num * wvslm["inc"] ** 0.9

# possible alternative: making education less important
wvslm["index_ses3"] = 0.9 * edu_num * wvslm["inc"]

**10.**

In [ ]:
import numpy as np

ms = wvslm["marital_stat"]
nc = wvslm["nmbr_chldrn"]
ph = wvslm["ppl_hh"]

wvslm["fam"] = np.select(
    [
        (ph == 0) & (ms == "single") & (nc == 0),
        (ms == "single") & ((nc != 0) | (ph != 0)),
        (ms == "married") & (nc == 0) & (ph == 1),
        (ms == "married") & (nc > 0) & (ph > 0),
    ],
    [0, 1, 2, 3],
    default=4,
)

wvslm.loc[ms.isna(), "fam"] = np.nan #R treats cases inside the conditional loop
# as NA, for Python we'll have to do it manually afterwards

**11.**

In [ ]:
wvslm.info()

print(wvslm.describe())

for col in wvslm.select_dtypes("category"):
    print(f"\n{col}:")
    print(wvslm[col].value_counts(dropna=False))